# Step-by-step process for applying PEFT to a pretrained model
This reading will guide you through the following steps:

Step 1: Prepare your dataset

Step 2: Fine-tune the pretrained model

Step 3: Evaluate the model

Step 4: Optimize the fine-tuning process (optional)

# Step 1: Prepare your dataset
The first step in fine-tuning a model is ensuring your dataset is appropriately prepared. In this walkthrough, we’ll follow the same steps from the activity to clean, tokenize, and split the dataset into training, validation, and test sets. This helps avoid overfitting and ensures proper model evaluation.

# Instructions
Load the dataset and inspect its structure.

Split the dataset into training, validation, and test sets to avoid overfitting and ensure good generalization.

Preprocess the data by cleaning, tokenizing, and padding the text.

Code example

### Recommended Dataset: AG News

For a text classification task involving fine-tuning a BERT model, a suitable dataset is the **AG News** dataset. It consists of news articles categorized into four classes: World, Sports, Business, and Sci/Tech. This dataset is commonly used for benchmarking text classification models.

In [1]:
import warnings
warnings.filterwarnings('ignore')

# Upgrade pip first to ensure proper dependency resolution
!pip install --upgrade pip

# Uninstall all potentially conflicting packages, including those that cause huggingface_hub conflicts
!pip uninstall -y pyarrow datasets huggingface_hub pandas fsspec transformers diffusers gradio peft accelerate

# Install specific compatible versions:
# datasets 2.19.1 requires huggingface_hub <1.0.0,>=0.17.0
# huggingface_hub 0.33.5 satisfies datasets requirement
# transformers 4.39.3 is pre-4.40.0 (where PreTrainedConfig moved)
# peft 0.7.1 is compatible with transformers 4.39.3
# accelerate 0.29.3 is compatible with transformers 4.39.3 and provides 'dispatch_batches'
!pip install datasets==2.19.1 huggingface_hub==0.33.5 transformers==4.39.3 peft==0.7.1 accelerate==0.29.3

# Install other necessary libraries, letting pip resolve their dependencies
!pip install --upgrade pyarrow pandas

from datasets import load_dataset
import pandas as pd

# Load the AG News dataset from the Hugging Face Hub
# The 'ag_news' dataset provides 'text' and 'label' columns, and labels are typically 0-indexed.
dataset = load_dataset('ag_news', split='train')

# Convert to pandas DataFrame for easier manipulation
df_agnews = dataset.to_pandas()

# The 'text' column already contains the combined title and description.
# The 'label' column is already 0-indexed (0, 1, 2, 3), so no adjustment is needed.

# Inspect the first few rows
display(df_agnews.head())

# Save the processed dataframe to a CSV file (this will be used by the splitting cell)
df_agnews[['text', 'label']].to_csv('ag_news_dataset.csv', index=False)
print("AG News dataset processed and saved as 'ag_news_dataset.csv'")

Found existing installation: pyarrow 24.0.0
Uninstalling pyarrow-24.0.0:
  Successfully uninstalled pyarrow-24.0.0
Found existing installation: datasets 2.19.1
Uninstalling datasets-2.19.1:
  Successfully uninstalled datasets-2.19.1
Found existing installation: huggingface-hub 0.33.5
Uninstalling huggingface-hub-0.33.5:
  Successfully uninstalled huggingface-hub-0.33.5
Found existing installation: pandas 3.0.4
Uninstalling pandas-3.0.4:
  Successfully uninstalled pandas-3.0.4
Found existing installation: fsspec 2024.3.1
Uninstalling fsspec-2024.3.1:
  Successfully uninstalled fsspec-2024.3.1
Found existing installation: transformers 4.39.3
Uninstalling transformers-4.39.3:
  Successfully uninstalled transformers-4.39.3
Found existing installation: peft 0.7.1
Uninstalling peft-0.7.1:
  Successfully uninstalled peft-0.7.1
Found existing installation: accelerate 0.29.3
Uninstalling accelerate-0.29.3:
  Successfully uninstalled accelerate-0.29.3
  Using cached datasets-2.19.1-py3-none-any.

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

,text,label
0,Wall St. Bears Claw Back Into the Black (Reute...,2
1,Carlyle Looks Toward Commercial Aerospace (Reu...,2
2,Oil and Economy Cloud Stocks' Outlook (Reuters...,2
3,Iraq Halts Oil Exports from Main Southern Pipe...,2
4,"Oil prices soar to all-time record, posing new...",2


AG News dataset processed and saved as 'ag_news_dataset.csv'


In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Load the dataset (using the AG News dataset saved previously)
data = pd.read_csv('ag_news_dataset.csv')

# Split dataset into training (70%), validation (15%), and test (15%)
train_data, temp_data = train_test_split(data, test_size=0.3, random_state=42)
val_data, test_data = train_test_split(temp_data, test_size=0.5, random_state=42)

# Reset indices to ensure contiguous integer indexing for the Trainer
train_data = train_data.reset_index(drop=True)
val_data = val_data.reset_index(drop=True)
test_data = test_data.reset_index(drop=True)

print(f"Training set size: {len(train_data)}")
print(f"Validation set size: {len(val_data)}")
print(f"Test set size: {len(test_data)}")

Training set size: 84000
Validation set size: 18000
Test set size: 18000


In [3]:
import pandas as pd

data = pd.read_csv('ag_news_dataset.csv')

print(data.columns.tolist())
print(data['label'].value_counts())  # or 'labels', 'class', etc.

['text', 'label']
label
2    30000
3    30000
1    30000
0    30000
Name: count, dtype: int64


# Explanation
Here, we use train_test_split to divide the dataset into three parts: 70 percent for training, 15 percent for validation, and 15 percent for testing. This is accomplished by re-splitting the temp_data output from the first call of train_test_split. The training set is used to fine-tune the model, the validation set helps to monitor training progress, and the test set evaluates the final model performance.

#Step 2: Fine-tune the pretrained model
Once the data is prepared, the next step is to fine-tune the pretrained model (e.g., BERT) using the training data. Fine-tuning involves adapting the model’s existing knowledge to a new, task-specific dataset.

Instructions
Load a pretrained model (e.g., BERT).

Set up the training arguments, such as batch size, learning rate, and number of epochs.

Begin fine-tuning the model on the training set.

Code example

In [4]:
import transformers
print(transformers.__version__)  # Should be 4.40+

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

4.39.3


In [5]:
from transformers import BertForSequenceClassification, Trainer, TrainingArguments, AutoTokenizer
from datasets import Dataset

# Load pretrained BERT model with the correct number of labels (4 for AG News)
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=4)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

# Define tokenization function
# - 'labels' key required by Trainer (not 'label')
# - labels are already 0-indexed (0-3) from our dataset, no adjustment needed
# - max_length=128 avoids defaulting to 512, which wastes memory and slows training
def tokenize_function(examples):
    result = tokenizer(
        examples['text'],
        padding='max_length',
        truncation=True,
        max_length=128
    )
    result['labels'] = examples['label']  # Trainer requires 'labels' column
    return result

# Convert pandas DataFrames to datasets.Dataset objects
# reset_index(drop=True) already done in the split cell, so no extra index column
train_dataset = Dataset.from_pandas(train_data)
val_dataset = Dataset.from_pandas(val_data)

# Tokenize the datasets
tokenized_train_dataset = train_dataset.map(tokenize_function, batched=True)
tokenized_val_dataset = val_dataset.map(tokenize_function, batched=True)

# Set up training arguments
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    evaluation_strategy="epoch", # Changed from eval_strategy to evaluation_strategy
    logging_dir='./logs',
    logging_steps=500,
)

# Fine-tune the model using the Trainer API
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_val_dataset,
)

# Start fine-tuning the model
trainer.train()

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Map:   0%|          | 0/84000 [00:00<?, ? examples/s]

Map:   0%|          | 0/18000 [00:00<?, ? examples/s]

wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: munyariri94 (munyariri94-byu-idaho) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch,Training Loss,Validation Loss
1,0.220800,0.209553
2,0.156200,0.204280


Epoch,Training Loss,Validation Loss
1,0.220800,0.209553
2,0.156200,0.204280
3,0.083600,0.238368


TrainOutput(global_step=15750, training_loss=0.16754440416608538, metrics={'train_runtime': 6864.3214, 'train_samples_per_second': 36.712, 'train_steps_per_second': 2.294, 'total_flos': 1.6576294146048e+16, 'train_loss': 0.16754440416608538, 'epoch': 3.0})

#Explanation
We use BERT for sequence classification, with three output labels in this example. The training arguments are defined using the TrainingArguments class, where you can specify the batch size, number of epochs, and evaluation strategy. Trainer is used to handle the fine-tuning process by taking care of the training loop, backpropagation, and model updates.

#Step 3: Evaluate the model
After fine-tuning the model, it’s time to evaluate its performance on the test set. This allows you to measure how well the model generalizes to new, unseen data.

Instructions
Evaluate the fine-tuned model on the test set using standard metrics such as accuracy, F1 score, precision, and recall.

Analyze the results to determine how well the model performs.

Code example

In [6]:
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from datasets import Dataset

# Convert test_data (pandas DataFrame) to a tokenized Dataset for the Trainer
test_dataset = Dataset.from_pandas(test_data)
tokenized_test_dataset = test_dataset.map(tokenize_function, batched=True)

# Evaluate the model on the test set
predictions_output = trainer.predict(tokenized_test_dataset)
predictions = predictions_output.predictions.argmax(axis=-1)

# Ground-truth labels from the original test split
true_labels = test_data['label'].tolist()

# Compute evaluation metrics
accuracy = accuracy_score(true_labels, predictions)
f1 = f1_score(true_labels, predictions, average='weighted')
precision = precision_score(true_labels, predictions, average='weighted')
recall = recall_score(true_labels, predictions, average='weighted')

print(f"Test Accuracy: {accuracy}")
print(f"Test F1 Score: {f1}")
print(f"Test Precision: {precision}")
print(f"Test Recall: {recall}")


Map:   0%|          | 0/18000 [00:00<?, ? examples/s]

Test Accuracy: 0.9467222222222222
Test F1 Score: 0.9467680757847221
Test Precision: 0.9468878459620084
Test Recall: 0.9467222222222222


In [ ]:
#saving the trained model and tokenizer
model.save_pretrained('./bert_agnews_model')
tokenizer.save_pretrained('./bert_agnews_model')
print("Model saved.")

In [8]:
# Save model and tokenizer
model.save_pretrained('./bert_agnews_model')
tokenizer.save_pretrained('./bert_agnews_model')
print("Model saved.")

Model saved.


In [9]:
from google.colab import drive
drive.mount('/content/drive')

model.save_pretrained('/content/drive/MyDrive/bert_agnews_model')
tokenizer.save_pretrained('/content/drive/MyDrive/bert_agnews_model')

Mounted at /content/drive


('/content/drive/MyDrive/bert_agnews_model/tokenizer_config.json',
 '/content/drive/MyDrive/bert_agnews_model/special_tokens_map.json',
 '/content/drive/MyDrive/bert_agnews_model/vocab.txt',
 '/content/drive/MyDrive/bert_agnews_model/added_tokens.json',
 '/content/drive/MyDrive/bert_agnews_model/tokenizer.json')

#Explanation
The model’s performance is evaluated using four metrics: accuracy, F1 score, precision, and recall. Accuracy gives a general overview of how well the model predicts, while F1 score, precision, and recall provide a deeper analysis of the model's effectiveness, especially for unbalanced datasets.

#Step 4: Optimize the fine-tuning process (optional)
You can experiment with different training configurations and hyperparameters to further improve the model's performance. This includes adjusting the learning rate, batch size, and number of epochs. You may also use hyperparameter search techniques to find the best configuration.

Optional instructions
Adjust hyperparameters such as learning rate and batch size.

Use a hyperparameter search to find the best settings for your task.

Code example

In [ ]:
#Then load from Drive next time:
model = BertForSequenceClassification.from_pretrained('/content/drive/MyDrive/bert_agnews_model')
tokenizer = AutoTokenizer.from_pretrained('/content/drive/MyDrive/bert_agnews_model')

#Explanation
Hyperparameter tuning allows you to experiment with different configurations, automatically finding the best setup for fine-tuning based on performance metrics.